# 34 — Task 2: benchmark mAP trên 26 nhãn

Task 2 dùng **26 nhãn**, là **tập con thật sự** của 40 nhãn Task 1. 14 nhãn bị bỏ:
`Adenopathy, Azygos Lobe, Clavicle Fracture, Fissure, Granuloma, Hydropneumothorax, Infarction,
Kyphosis, Lobar Atelectasis, Pulmonary Embolism, Pulmonary Hypertension, Rib Fracture,
Round(ed) Atelectasis, Tuberculosis`.

**Hai tập chấm, và chúng KHÔNG so được với nhau:**

| tập | số ảnh | prevalence TB | ghi chú |
|---|---|---|---|
| `dev` (`development_labeled_task2.csv`) | **39,293** | 9.13% | **trùng khít** tập val của ta ⇒ so được với mọi số ở §16–17 |
| `test` (`test_labeled_task2.csv`) | **406** | **16.10%** | tập chính thức của Task 2 ⇒ **chỉ tập này** mới so được với mốc 0.526 |

**Vì sao phải tách bạch.** AP của một bộ đoán ngẫu nhiên **bằng đúng prevalence**. Tập 406 ảnh được
làm giàu lớp hiếm rất mạnh — `Pneumoperitoneum` **43×**, `Pneumomediastinum` **28×**,
`Pleural Other` **23×**, `Fibrosis` **13×** so với dev. Sàn ngẫu nhiên vì thế nhảy từ `0.091` lên
`0.161`. Lấy số dev đem so với `0.526` là so hai thang đo khác nhau.

**Hai câu hỏi phụ trả lời luôn:**

- huấn luyện trên **40 nhãn** rồi cắt lấy 26, so với huấn luyện thẳng **26 nhãn** — 14 nhãn kia là
  nhiệm vụ phụ có ích hay chỉ là nhiễu?
- arm/biểu diễn tốt nhất cho Task 2 có trùng với Task 1 (`nosq518/patch`) không?

In [ ]:
import os, io, json, time, zipfile, itertools
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/MIMIC_CXR'; DATA_DIR = os.path.join(ROOT, 'planB_out')
except Exception:
    ROOT, DATA_DIR = '.', './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES_DIR = os.path.join(OUT_DIR, 'stage3_baselines')
T2_DIR  = os.path.join(OUT_DIR, 'task2'); os.makedirs(T2_DIR, exist_ok=True)

W_FRONTAL, W_LATERAL = 0.7, 0.3
SEED = 42
TIER_EDGES = (1000, 10000)          # giong 28: tail < 1000 <= medium < 10000 <= head

# Tap 406 anh la TEST. Ke hoach quy dinh chi cham vao test dung mot lan o cuoi.
# Day la mot lan do, KHONG duoc dung de chon arm/nguong/sieu tham so.
SCORE_TEST = True

# --- doc file cua CXR-LT: uu tien thu muc da giai nen, khong thi doc thang trong zip ---
ZIP  = os.path.join(ROOT, 'cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0.zip')
BASE = 'cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0/cxr-lt-2024'
DEST = ZIP[:-4]
def _open(name):
    p = os.path.join(DEST, BASE, name)
    if os.path.exists(p): return p
    assert os.path.exists(ZIP), f'khong thay ca {p} lan {ZIP}'
    return zipfile.ZipFile(ZIP).open(f'{BASE}/{name}')

ID_COLS = ['dicom_id', 'subject_id', 'study_id', 'ViewPosition',
           'ViewCodeSequence_CodeMeaning', 'fpath', 'split']

DEV2  = pd.read_csv(_open('development_labeled_task2.csv'))
TEST2 = pd.read_csv(_open('test_labeled_task2.csv'))
LAB26 = [c for c in DEV2.columns if c not in ID_COLS]

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y40 = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'),
                       columns=['dicom_id'] + LAB40)
         .set_index('dicom_id').loc[ORDER.dicom_id])
assert (Y40.index.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y40[LAB26].to_numpy('float32')            # nhan 26 cot, cung thu tu voi embedding

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
if os.path.exists(P_VIEW):
    V = (pd.read_parquet(P_VIEW)[['dicom_id', 'view_group', 'use']]
           .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
else:
    V = ORDER[['dicom_id', 'view_group']].copy(); V['use'] = V.view_group != 'unknown'

SPLIT = ORDER.split.to_numpy()
TR, VA, TE = SPLIT == 'train', SPLIT == 'val', SPLIT == 'test'
ROW = {d: i for i, d in enumerate(ORDER.dicom_id)}
ARMS  = [a for a in ('crop518', 'lb518', 'nosq518', 'lb1022')
         if os.path.exists(os.path.join(EMB_DIR, f'emb_{a}_cls.npy'))]
KINDS = ['cls', 'patch', 'both']
assert ARMS, 'chua co embedding nao -> chay 31 truoc'

print(f'{len(ORDER):,} anh | train {TR.sum():,} | val {VA.sum():,} | test {TE.sum():,}')
print(f'task2: {len(LAB26)} nhan | dev {len(DEV2):,} anh | test {len(TEST2):,} anh')
print(f'arm da co: {ARMS}')

## C1 — Cổng: hai tập của Task 2 nằm ở đâu trong split của ta

Bốn điều phải đúng trước khi chấm bất cứ thứ gì:

1. **26 ⊂ 40** — nếu không thì không cắt cột từ nhãn cũ được
2. **`dev` của Task 2 trùng khít tập `val` của ta** — nếu lệch thì mọi so sánh với §16–17 vô nghĩa
3. **`test` 406 ảnh nằm trong split `test`** và **không dính bệnh nhân nào** của dev
4. **giá trị nhãn khớp** giữa `labels.csv` (nguồn ta dùng) và file task2 — hai file khác nhau, phải
   chứng minh chúng nói cùng một chuyện chứ không giả định

Điều 4 đúng cho `dev` (0 ô lệch) nhưng **KHÔNG đúng cho tập 406 ảnh** — và đó là chủ đích, không
phải lỗi. `labels.csv` là nhãn **tự động trích từ báo cáo**; 406 ảnh của Task 2 được **gán nhãn
lại**. Cell in mức lệch và chiều lệch (thêm hay bỏ tổn thương) thay vì bắt lỗi. `TEST2` là nguồn
chuẩn cho tập đó.

Hệ quả phải nhớ khi đọc C4: **ta huấn luyện trên nhãn tự động, chấm trên nhãn chuẩn.** Nhà vô địch
cũng vậy nên so sánh vẫn công bằng, nhưng mAP trên test406 chịu thêm một nguồn sai lệch mà mAP trên
dev không có.

Cell cũng in bảng prevalence hai tập. Đó **không phải** thông tin phụ: sàn ngẫu nhiên của AP bằng
đúng prevalence, nên bảng này là thứ giải thích vì sao hai tập không so được.

In [ ]:
print(f'=== cong 1: 26 nhan co phai tap con cua 40 ===')
assert set(LAB26) <= set(LAB40), f'nhan la: {sorted(set(LAB26) - set(LAB40))}'
print(f'  OK | 14 nhan bi bo: {sorted(set(LAB40) - set(LAB26))}')

print(f'\n=== cong 2: dev task2 vs tap val cua ta ===')
_va_id = set(ORDER.dicom_id.to_numpy()[VA]); _dev_id = set(DEV2.dicom_id)
print(f'  val {len(_va_id):,} | dev {len(_dev_id):,} | giao {len(_va_id & _dev_id):,}')
_same_dev = _va_id == _dev_id
print(f'  {"OK - trung khit" if _same_dev else "LECH"}')
assert _same_dev, 'dev task2 KHAC tap val -> khong so duoc voi §16-17'

print(f'\n=== cong 3: test 406 anh ===')
_te_id = set(ORDER.dicom_id.to_numpy()[TE]); _t2_id = set(TEST2.dicom_id)
print(f'  {len(_t2_id):,} anh | nam trong split test: {_t2_id <= _te_id}')
assert _t2_id <= _te_id, 'co anh cua test task2 khong nam trong split test'
_ov = set(DEV2.subject_id) & set(TEST2.subject_id)
print(f'  benh nhan trung voi dev: {len(_ov)}')
assert not _ov, f'{len(_ov)} benh nhan xuat hien o CA dev lan test'

print(f'\n=== cong 4: gia tri nhan khop giua labels.csv va file task2 ===')
_d = DEV2.set_index('dicom_id')[LAB26].loc[ORDER.dicom_id.to_numpy()[VA]].to_numpy('float32')
_n = int((_d != YM[VA]).sum())
print(f'  o lech tren dev: {_n} / {_d.size:,}')
assert _n == 0, f'{_n} o nhan KHAC nhau giua hai nguon'

# --- tap 406: TEST2 la NGUON CHUAN, khong phai labels.csv ---
# Lech so voi labels.csv la CHUYEN DE, khong phai loi. 406 anh nay duoc gan nhan lai
# (nhieu kha nang boi chuyen gia); labels.csv la nhan tu dong trich tu bao cao — dev thi
# hai nguon trung khit, rieng tap nay thi khong. Ta HUAN LUYEN tren nhan tu dong va CHAM
# tren nhan chuan; do chinh la thiet ke cua Task 2, va giai thich vi sao tap nay nho va
# giau lop hiem (chon ca co ton thuong hiem roi gan tay).
assert TEST2.dicom_id.is_unique, 'dicom_id trong test_labeled_task2 KHONG duy nhat'
T406 = np.array([ROW[d] for d in TEST2.dicom_id])
Y406 = TEST2[LAB26].to_numpy('float32')
_dl = (Y406 != YM[T406])
print(f'  lech so voi labels.csv: {_dl.sum():,}/{_dl.size:,} = {_dl.mean():.2%} '
      f'| them ton thuong {int(((YM[T406]==0)&(Y406==1)).sum()):,} '
      f'| bo ton thuong {int(((YM[T406]==1)&(Y406==0)).sum()):,}')
assert _dl.mean() < 0.5, 'lech qua nua so o -> nghi LECH HANG chu khong phai gan nhan lai'
print(f'  OK | tap {len(T406)} anh -> {len(set(ORDER.study_id.to_numpy()[T406])):,} study')

# --- prevalence: day la thu giai thich vi sao hai tap khong so duoc ---
PV = pd.DataFrame({
    'n_train':  YM[TR].sum(0).astype(int),
    'dev_%':    YM[VA].mean(0) * 100,
    'test406_%': Y406.mean(0) * 100}, index=LAB26)
PV['lam_giau'] = PV['test406_%'] / PV['dev_%'].replace(0, np.nan)
PV['tier'] = np.where(PV.n_train >= TIER_EDGES[1], 'head',
              np.where(PV.n_train >= TIER_EDGES[0], 'medium', 'tail'))
print('\n=== PREVALENCE: dev vs test406 ===')
print(PV.sort_values('lam_giau', ascending=False)
        .to_string(float_format=lambda v: f'{v:.2f}'))
_rd, _rt = YM[VA].mean(), Y406.mean()
print(f'''
san NGAU NHIEN cua mAP (= prevalence trung binh):
  dev      {_rd:.4f}
  test406  {_rt:.4f}   ({_rt/_rd:.2f}x cao hon)
Moi so tren dev PHAI so voi {_rd:.4f}, moi so tren test406 phai so voi {_rt:.4f}.
Moc 0.526 cua nha vo dich la tren test406 — dung dem so dev ra so voi no.''')
PV.to_csv(os.path.join(T2_DIR, 'task2_prevalence.csv'))
TIER26 = PV.tier.to_dict()
print('\nphan tang 26 nhan:', PV.tier.value_counts().to_dict())

## C2 — Khớp probe 26 đầu ra, dự đoán cả `val` lẫn `test`

Cùng cấu hình với `32` (một lớp tuyến tính, BCE, 30 epoch, `lr 3e-3`, `wd 1e-4`, chuẩn hoá **chỉ
trên train**) — chỉ khác số đầu ra: **26** thay vì 40.

Một lượt khớp cho ra dự đoán cho **cả hai** tập, nên không phải khớp lại lần nữa cho `test`.

Có RESUME theo `(arm, kind)`. **Nếu bạn thay đổi embedding của một arm thì phải xoá dòng của nó
trong `task2_probe.csv` VÀ xoá `q_<arm>_<kind>.npz`** — RESUME chỉ khoá theo tên, không nhìn tới
file embedding (đúng cái bẫy đã dính ở `32`).

In [ ]:
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'thiet bi: {dev}')

def fit_probe(Xtr, ytr, Xs, epochs=30, lr=3e-3, wd=1e-4, bs=4096):
    '''Linear(D->C) + BCE. Tra ve list xac suat, moi phan tu ung voi mot X trong Xs.'''
    torch.manual_seed(SEED)
    mu, sd = Xtr.mean(0, keepdims=True), Xtr.std(0, keepdims=True) + 1e-6   # CHI train
    xt = torch.from_numpy((Xtr - mu) / sd).to(dev)
    yt = torch.from_numpy(ytr).to(dev)
    m   = nn.Linear(xt.shape[1], yt.shape[1]).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    lossf = nn.BCEWithLogitsLoss(); n = len(xt)
    for ep in range(epochs):
        perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            opt.zero_grad(set_to_none=True)
            lossf(m(xt[idx]), yt[idx]).backward(); opt.step()
        sch.step()
    out = []
    with torch.no_grad():
        for X in Xs:
            out.append(torch.sigmoid(m(torch.from_numpy((X - mu) / sd).to(dev))).cpu().numpy())
    return out

def feats(arm, kind):
    c = np.load(os.path.join(EMB_DIR, f'emb_{arm}_cls.npy'),   mmap_mode='r')
    p = np.load(os.path.join(EMB_DIR, f'emb_{arm}_patch.npy'), mmap_mode='r')
    if kind == 'cls':   return np.asarray(c)
    if kind == 'patch': return np.asarray(p)
    return np.concatenate([np.asarray(c), np.asarray(p)], 1)

P_CSV = os.path.join(T2_DIR, 'task2_probe.csv')
done = set()
if os.path.exists(P_CSV):
    _d = pd.read_csv(P_CSV); done = set(zip(_d.arm, _d.kind))
    print(f'resume: da co {len(done)} to hop')

for arm, kind in itertools.product(ARMS, KINDS):
    f = os.path.join(T2_DIR, f'q_{arm}_{kind}.npz')
    if (arm, kind) in done and os.path.exists(f): continue
    t0 = time.time(); X = feats(arm, kind)
    q_va, q_te = fit_probe(X[TR], YM[TR], [X[VA], X[T406]])
    del X
    np.savez_compressed(f, val=q_va.astype('float32'), test406=q_te.astype('float32'))
    pd.DataFrame([dict(arm=arm, kind=kind)]).to_csv(
        P_CSV, mode='a', index=False, header=not os.path.exists(P_CSV))
    print(f'  {arm:<8} {kind:<5} xong trong {time.time()-t0:.0f}s')

Q = {}
for arm, kind in itertools.product(ARMS, KINDS):
    f = os.path.join(T2_DIR, f'q_{arm}_{kind}.npz')
    if os.path.exists(f):
        z = np.load(f); Q[(arm, kind)] = (z['val'], z['test406'])
print(f'\nco du doan cho {len(Q)} to hop')

## C3 — mAP trên `dev` (39,293 ảnh), và phần thưởng của 14 nhãn phụ

`slice40` = lấy dự đoán 40 nhãn đã lưu ở `32` rồi **cắt 26 cột**. `native26` = probe khớp thẳng
trên 26 nhãn. Hiệu số giữa hai cái trả lời: **14 nhãn kia là nhiệm vụ phụ có ích hay chỉ là nhiễu?**

Với một lớp tuyến tính + BCE, mỗi đầu ra có hàng trọng số riêng và mất mát tính riêng từng nhãn —
ghép nối duy nhất là qua `weight_decay` và lịch học chung. Nên hiệu số này **phải gần 0**; nếu
không thì có gì đó đáng đào.

In [ ]:
def ap_rank(y, s):
    '''AP: sum (R_n - R_{n-1}) * P_n tren cac NGUONG PHAN BIET (diem bang nhau gop lam mot).'''
    n1 = y.sum()
    if n1 == 0 or n1 == len(y): return np.nan
    o = np.argsort(-s, kind='stable'); yy, ss = y[o], s[o]
    tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
    last = np.r_[np.diff(ss) != 0, True]
    tp, fp = tp[last], fp[last]
    prec, rec = tp / (tp + fp), tp / n1
    return float((np.diff(np.r_[0.0, rec]) * prec).sum())

def ap_all(y, q): return np.array([ap_rank(y[:, j], q[:, j]) for j in range(y.shape[1])])

_WV = {'frontal': W_FRONTAL, 'lateral': W_LATERAL}
def pool_study(q, rows):
    '''0.7 frontal + 0.3 lateral tren DU DOAN -> (nhan muc study, du doan muc study).'''
    d = pd.DataFrame(q.astype('float64'), columns=LAB26)
    d['sid'] = ORDER.study_id.to_numpy()[rows]
    d['vg']  = V.view_group.to_numpy()[rows]
    d['use'] = V.use.to_numpy()[rows]
    d = d[d.use & d.vg.isin(_WV)]
    g1 = d.groupby(['sid', 'vg'], sort=True)[LAB26].mean().reset_index()
    g1['w'] = g1.vg.map(_WV)
    out = g1[LAB26].mul(g1.w, axis=0).groupby(g1.sid).sum().div(g1.groupby('sid').w.sum(), axis=0)
    ys = (pd.DataFrame(YM[rows], columns=LAB26).assign(sid=ORDER.study_id.to_numpy()[rows])
            .groupby('sid')[LAB26].max().loc[out.index])
    return ys.to_numpy('float32'), out.to_numpy('float64')

_va_rows = np.flatnonzero(VA)
_j40 = [LAB40.index(l) for l in LAB26]          # cot cua 26 nhan trong bang 40 nhan

rows = []
for (arm, kind), (q_va, _) in sorted(Q.items()):
    a_i = ap_all(YM[VA], q_va)
    ys, ps = pool_study(q_va, _va_rows)
    a_s = ap_all(ys, ps)
    r = dict(arm=arm, kind=kind, bien_the='native26',
             mAP_anh=np.nanmean(a_i), mAP_study=np.nanmean(a_s))
    f40 = os.path.join(RES_DIR, f'p_{arm}_{kind}.npy')
    if os.path.exists(f40):                      # mien phi: cat 26 cot tu du doan 40 nhan cua 32
        p = np.load(f40)[:, _j40]
        r['mAP_anh_slice40'] = np.nanmean(ap_all(YM[VA], p))
        ys2, ps2 = pool_study(p, _va_rows)
        r['mAP_study_slice40'] = np.nanmean(ap_all(ys2, ps2))
    rows.append({**r, **{f'ap_{l}': v for l, v in zip(LAB26, a_i)}})
D = pd.DataFrame(rows)

_c = ['mAP_anh', 'mAP_study'] + [c for c in ('mAP_anh_slice40', 'mAP_study_slice40') if c in D]
print('=== mAP tren DEV (26 nhan, 39,293 anh) ===')
print(D.pivot(index='arm', columns='kind', values='mAP_anh').reindex(columns=KINDS)
       .to_string(float_format=lambda v: f'{v:.4f}'))
print('\n=== mAP muc STUDY ===')
print(D.pivot(index='arm', columns='kind', values='mAP_study').reindex(columns=KINDS)
       .to_string(float_format=lambda v: f'{v:.4f}'))
if 'mAP_anh_slice40' in D:
    D['aux40'] = D.mAP_anh_slice40 - D.mAP_anh
    print('\n=== huan luyen 40 nhan roi cat 26, TRU huan luyen thang 26 (muc anh) ===')
    print(D.pivot(index='arm', columns='kind', values='aux40').reindex(columns=KINDS)
           .to_string(float_format=lambda v: f'{v:+.4f}'))
    print(f'  lech lon nhat {D.aux40.abs().max():.4f} — duoi 0.005 la nhieu, tuc 14 nhan phu '
          f'khong them cung khong bot gi (dung nhu ky vong voi lop tuyen tinh).')
BEST = D.loc[D.mAP_anh.idxmax()]
print(f'\ntot nhat tren dev: {BEST.arm} / {BEST.kind} — anh {BEST.mAP_anh:.4f}, '
      f'study {BEST.mAP_study:.4f}')
D.to_csv(os.path.join(T2_DIR, 'task2_dev.csv'), index=False)

## C4 — `test` 406 ảnh: tập DUY NHẤT so được với mốc 0.526

Đây là tập chính thức của Task 2. Ba điều phải nhớ khi đọc:

**Sàn ngẫu nhiên cao gấp ~1.76 lần dev** (0.161 vs 0.091) vì tập được làm giàu lớp hiếm. Một phần
lớn chênh lệch giữa 0.526 và số dev của ta là **cơ học**, không phải chất lượng mô hình. Cell in
`mAP / san ngau nhien` để khử phần đó.

**406 ảnh là rất ít.** Nhãn hiếm nhất chỉ có 6 ca dương (`Lung Lesion`). AP trên vài ca dương có
phương sai khổng lồ — không được đọc chênh lệch nhỏ giữa các arm ở đây.

**Vô địch dùng fine-tune + ensemble**, ta dùng probe đông cứng. Khoảng cách là dự kiến; điều đáng
đọc là **tỉ lệ**, không phải hiệu số.

In [ ]:
if not SCORE_TEST:
    print('SCORE_TEST = False -> bo qua. Dat True o C0 neu muon cham tap 406 anh.')
else:
    _log = os.path.join(T2_DIR, '_test_touches.jsonl')
    with open(_log, 'a') as _f:
        _f.write(json.dumps(dict(n=len(T406), arms=ARMS, kinds=KINDS)) + '\n')
    _nt = sum(1 for _ in open(_log))
    print(f'''!! Dang cham tren TEST. Day la lan thu {_nt} tap 406 anh nay duoc doc
   (nhat ky: {_log}). Khong duoc dung ket qua nay de chon arm / nguong / sieu tham so
   — chi de BAO CAO. Moi lua chon phai quyet tren dev.
''')
    _te_rows = T406
    rt = []
    for (arm, kind), (_, q_te) in sorted(Q.items()):
        a_i = ap_all(Y406, q_te)
        ys, ps = pool_study(q_te, _te_rows)
        a_s = ap_all(ys, ps)
        rt.append(dict(arm=arm, kind=kind, mAP_anh=np.nanmean(a_i),
                       mAP_study=np.nanmean(a_s), n_nhan=int((~np.isnan(a_i)).sum()),
                       **{f'ap_{l}': v for l, v in zip(LAB26, a_i)}))
    Tt = pd.DataFrame(rt)
    print('=== mAP tren TEST 406 anh (26 nhan) — muc ANH ===')
    print(Tt.pivot(index='arm', columns='kind', values='mAP_anh').reindex(columns=KINDS)
            .to_string(float_format=lambda v: f'{v:.4f}'))
    print('\n=== muc STUDY ===')
    print(Tt.pivot(index='arm', columns='kind', values='mAP_study').reindex(columns=KINDS)
            .to_string(float_format=lambda v: f'{v:.4f}'))

    _san_te, _san_dv = float(Y406.mean()), float(YM[VA].mean())
    B = Tt.loc[Tt.mAP_anh.idxmax()]
    _bd = D[D.arm.eq(BEST.arm) & D.kind.eq(BEST.kind)].iloc[0]
    print(f'''
=== DOI CHIEU VOI MOC CUA GIAI ===
  Task 2 vo dich (test 406 anh, fine-tune + ensemble)   0.5260
  ta, tot nhat tren test  ({B.arm}/{B.kind})            {B.mAP_anh:.4f}   = {B.mAP_anh/0.526:.0%} cua vo dich
  ta, tot nhat tren dev   ({BEST.arm}/{BEST.kind})      {_bd.mAP_anh:.4f}   <- KHONG so voi 0.526 duoc

=== KHU PHAN CO HOC CUA PREVALENCE (mAP / san ngau nhien) ===
  test406  {B.mAP_anh:.4f} / {_san_te:.4f} = {B.mAP_anh/_san_te:.2f}x san
  dev      {_bd.mAP_anh:.4f} / {_san_dv:.4f} = {_bd.mAP_anh/_san_dv:.2f}x san
  vo dich  0.5260 / {_san_te:.4f} = {0.526/_san_te:.2f}x san

Doc theo BOI SO SAN thi hai tap moi so duoc voi nhau. Neu boi so tren test406 xap xi tren dev
thi mo hinh khong he "kem hon o Task 2" — chi la thang do khac.''')
    Tt.to_csv(os.path.join(T2_DIR, 'task2_test406.csv'), index=False)

## C4-bis — Trần do nhiễu nhãn, và nhãn nào ta nhìn xuyên qua được

Ta **huấn luyện trên nhãn tự động** (NLP trích từ báo cáo) nhưng **chấm trên nhãn chuyên gia**.
Trên 406 ảnh này chuyên gia **bỏ nhiều gấp ~4 lần số lần thêm** — nhãn tự động **gọi thừa**.

Nên có một mốc tham chiếu tự nhiên: **lấy chính nhãn tự động làm dự đoán**, chấm trên nhãn chuyên
gia. Đó là AP mà **tín hiệu huấn luyện** đạt được. Nó trả lời "phần nào của khoảng cách tới 0.526
là do nhiễu nhãn chứ không phải do mô hình nhìn kém".

**Đây KHÔNG phải trần tuyệt đối.** Mô hình cho điểm liên tục nên vẫn có thể xếp hạng tốt hơn chính
nhãn nhị phân đã dạy nó: nếu các ca dương-tính-giả không có dấu hiệu hình ảnh thật, mô hình sẽ tự
cho chúng điểm thấp. **Nhãn nào ta vượt mốc này là nhãn ta nhìn xuyên qua được nhiễu nhãn** — và
đó mới là thứ đáng đọc, chứ không phải mAP thô.

In [ ]:
if SCORE_TEST:
    _auto = YM[T406]                        # nhan tu dong (labels.csv) — THU DA DAY mo hinh
    ref = ap_all(Y406, _auto)               # AP cua chinh tin hieu huan luyen
    _mine = Tt[Tt.arm.eq(B.arm) & Tt.kind.eq(B.kind)][[f'ap_{l}' for l in LAB26]].iloc[0].to_numpy('float64')
    Z = pd.DataFrame(dict(label=LAB26, tran_nhan=ref, ta=_mine))
    Z['vuot'] = Z.ta - Z.tran_nhan
    Z['auto_1']  = _auto.sum(0).astype(int)
    Z['chuyengia_1'] = Y406.sum(0).astype(int)
    Z['bo']   = ((_auto == 1) & (Y406 == 0)).sum(0).astype(int)
    Z['them'] = ((_auto == 0) & (Y406 == 1)).sum(0).astype(int)
    Z['tier'] = Z.label.map(TIER26)
    print(f'=== TRAN DO NHIEU NHAN tren test406 ({B.arm}/{B.kind}) ===')
    print(Z.sort_values('tran_nhan').to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    _nv = int((Z.vuot > 0).sum())
    print(f'''
  AP cua chinh nhan huan luyen (macro)  {ref.mean():.4f}
  ta                                    {_mine.mean():.4f}  = {_mine.mean()/ref.mean():.0%}
  vo dich Task 2                        0.5260  = {0.526/ref.mean():.0%}

  ta VUOT moc nay o {_nv}/{len(LAB26)} nhan -> o do mo hinh NHIN XUYEN QUA nhieu nhan,
  chu khong chi tai tao lai no. Duoi moc o {len(LAB26)-_nv} nhan.

  Nhan bi tran keo xuong thap nhat deu la GOI THUA thuan tuy (cot `them` = 0):''')
    print(Z.nsmallest(5, 'tran_nhan')[['label','auto_1','chuyengia_1','bo','them','tran_nhan','ta']]
           .to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print(f'''
  Ham y cho Stage 4: voi Task 2, LAM SACH NHAN co the la don bay lon hon THI GIAC.
  Ba nhan {", ".join(Z.nsmallest(3, "tran_nhan").label)} khong the vuot
  {Z.nsmallest(3, "tran_nhan").tran_nhan.max():.3f} neu con hoc tu nhan tu dong nguyen ban.''')
    Z.to_csv(os.path.join(T2_DIR, 'task2_label_ceiling.csv'), index=False)
else:
    print('can SCORE_TEST = True')

## C5 — Từng nhãn, và phân tầng tính lại trên 26 nhãn

Phân tầng head/medium/tail **tính lại** theo `n_train` của chính 26 nhãn (ngưỡng 1,000 / 10,000
như `28`), không mượn phân tầng của 40 nhãn — bỏ 14 nhãn hiếm đi thì ranh giới dịch.

In [ ]:
_apc = [c for c in D.columns if c.startswith('ap_')]
_lbl = [c[3:] for c in _apc]
Tl = pd.DataFrame(dict(label=_lbl,
                       dev=D[D.arm.eq(BEST.arm) & D.kind.eq(BEST.kind)][_apc].iloc[0].to_numpy('float64')))
Tl['tier'] = Tl.label.map(TIER26)
Tl['n_train'] = Tl.label.map(PV.n_train.to_dict())
Tl['dev_pct'] = Tl.label.map(PV['dev_%'].to_dict())
if SCORE_TEST:
    Tl['test406'] = Tt[Tt.arm.eq(B.arm) & Tt.kind.eq(B.kind)][_apc].iloc[0].to_numpy('float64')
    Tl['test_pct'] = Tl.label.map(PV['test406_%'].to_dict())
    Tl['n_pos_406'] = (Tl.test_pct / 100 * len(T406)).round().astype(int)

print(f'=== tung nhan | dev: {BEST.arm}/{BEST.kind}'
      + (f' | test406: {B.arm}/{B.kind}' if SCORE_TEST else '') + ' ===')
print(Tl.sort_values('dev', ascending=False).to_string(index=False,
      float_format=lambda v: f'{v:.4f}'))

print('\n=== theo TIER (tinh lai tren 26 nhan) ===')
_ag = dict(n=('dev', 'size'), mAP_dev=('dev', 'mean'))
if SCORE_TEST: _ag['mAP_test406'] = ('test406', 'mean')
print(Tl.groupby('tier').agg(**_ag).reindex(['head', 'medium', 'tail'])
        .to_string(float_format=lambda v: f'{v:.4f}'))
print(f'''
MOC Task 1 de doi chieu (nosq518/patch, 40 nhan, dev):
  mAP 0.2028 | head 0.4204 | medium 0.1369 | tail 0.0417
Task 2 cao hon la DUONG NHIEN: no bo di 14 nhan hiem nhat, tuc bo di phan kho nhat.''')
Tl.to_csv(os.path.join(T2_DIR, 'task2_per_label.csv'), index=False)

## C6 — Tổng hợp

In [ ]:
print('=' * 68); print('TOM TAT 34 — TASK 2 (26 nhan)'); print('=' * 68)
print(f'''
1. HAI TAP, HAI THANG DO
   dev      {len(YM[VA]):>6,} anh | san ngau nhien {YM[VA].mean():.4f}
   test406  {len(Y406):>6,} anh | san ngau nhien {Y406.mean():.4f}  ({Y406.mean()/YM[VA].mean():.2f}x)
   Chi test406 moi so duoc voi moc 0.526.

2. TOT NHAT TREN DEV: {BEST.arm} / {BEST.kind}
   mAP anh {BEST.mAP_anh:.4f} | study {BEST.mAP_study:.4f}''')
if 'aux40' in D:
    print(f'''
3. 14 NHAN PHU (huan luyen 40 roi cat 26, tru huan luyen thang 26)
   lech lon nhat {D.aux40.abs().max():.4f} -> '''
          + ('khong dang ke, dung nhu ky vong voi lop tuyen tinh'
             if D.aux40.abs().max() < 0.005 else
             'VUOT nguong nhieu -> dang dao, khong nen coi la nhiem vu phu vo hai'))
if SCORE_TEST:
    print(f'''
4. TREN TEST 406 ANH (tap chinh thuc Task 2)
   ta {B.mAP_anh:.4f} ({B.arm}/{B.kind}) | vo dich 0.5260 | ta bang {B.mAP_anh/0.526:.0%}
   theo boi so san: ta {B.mAP_anh/Y406.mean():.2f}x | vo dich {0.526/Y406.mean():.2f}x
   406 anh la RAT IT — khong doc chenh lech nho giua cac arm o day.''')
print(f'''
Da ghi vao task2/:
  task2_prevalence.csv | task2_dev.csv | task2_per_label.csv'''
      + (' | task2_test406.csv' if SCORE_TEST else ''))

## C7 · Chấm lại Task 2 bằng **kiến trúc hiện tại** `t1_prior_both` — 0 giây GPU

Mọi số ở C3–C6 đến từ **probe tuyến tính** trên RAD-DINO đông cứng (thời 0.20). Kiến trúc đã chốt
từ `53` là `t1_prior_both` = ML-Decoder + `Tier1` (EHR) + `Tier1Img` (phim trước), val 40 nhãn
**0.2560** qua 2 seed.

Và không cần suy luận lại: cổng 2 đã chứng minh `dev Task 2 ≡ val của ta`, cổng 3 đã chứng minh
`406 ⊂ split test`. `53` đã lưu dự đoán cho **cả hai** tập. Nên chấm Task 2 bằng kiến trúc mới chỉ
là **cắt 26 cột** từ mảng có sẵn — không một giây GPU nào.

Ba con số đọc được:

```
dev  probe(C3)  -> t1_prior_both     kien truc mua duoc bao nhieu tren 39,293 anh
406  probe(C4)  -> t1_prior_both     ... va tren tap CHUYEN GIA
406  so voi 0.526 cua vo dich Task 2
```

**Đọc tập 406 phải kèm khoảng tin cậy.** 406 ảnh × 26 nhãn, nhiều nhãn chỉ vài ca dương ⇒ sàn
nhiễu ở đây lớn hơn sàn ±0.0007 của dev **hàng chục lần**. Bootstrap theo **bệnh nhân** ở dưới cho
con số thật; mọi hiệu số nhỏ hơn nó không đọc được, dù dương.

In [ ]:
import glob
# --- NGUON DU DOAN: doi ba dong duoi la chuyen sang kien truc khac, khong sua gi them ---
# Hien tai `53/t1_prior_both` = ML-Decoder + Tier1 (EHR) + Tier1Img (phim truoc), val 40 nhan
# 0.2560 qua 2 seed — kien truc CHOT DA XAC NHAN.
# Neu 56 cho `t1_prior_both_film` vuot (cong: G_deep > +0.004 VA tail khong tut) thi doi thanh
#     SRC = os.path.join(OUT_DIR, 'stage4_nb56', 'r1');  ARM = 't1_prior_both_film'
SRC  = os.path.join(OUT_DIR, 'stage4_nb53')
ARM  = 't1_prior_both'
EP, NLAST = 10, 3

IDX26  = np.array([LAB40.index(l) for l in LAB26])
I_VA53 = np.where(VA)[0]
I_TE53 = np.where(~(TR | VA))[0]
assert (I_TE53 == np.where(TE)[0]).all(), 'split test cua nguon khac 34 -> khong tai dung duoc'

if 'ap_all' not in dir():
    # C7 phai TU DUNG DUOC. `ap_all` von nam o C3, ma C3 lai can `Q` tu C2 (khop probe, ton phut);
    # bat C2 chay chi de lay mot ham la phu thuoc NGAM — dung thu vua khien 41c hong ba lan.
    def ap_all(Yc, Sc):
        o = np.full(Yc.shape[1], np.nan)
        for j in range(Yc.shape[1]):
            y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
            if n1 == 0 or n1 == len(y): continue
            r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
            tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
            last = np.r_[np.diff(ss) != 0, True]          # diem BANG NHAU gop lam mot nguong
            tp, fp = tp[last], fp[last]
            pr, rc = tp / (tp + fp), tp / n1
            o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
        return o
    from sklearn.metrics import average_precision_score as _aps
    _r6 = np.random.default_rng(6); _y6 = (_r6.random((2000, 6)) < 0.2).astype('float32')
    _s6 = _r6.random((2000, 6)).astype('float32')
    assert np.nanmax(np.abs(ap_all(_y6, _s6) - np.array(
        [_aps(_y6[:, j], _s6[:, j]) for j in range(6)]))) < 1e-6, 'ap_all du phong lech sklearn'
    print('C3 chua chay -> dung `ap_all` du phong (da doi chieu sklearn toi 1e-6)')

# ---------- DEV = tap val cua ta (cong 2 da chung minh trung khit) ----------
# `53` ghi vao THU MUC CON theo RUN_TAG (stage4_nb53/<tag>/), khong phai ngay duoi stage4_nb53.
# Do de quy roi in ra thu thay duoc — dung bat nguoi doc tu doan tang thu muc.
def _find(pat):
    f = sorted(glob.glob(os.path.join(SRC, pat)))
    return f if f else sorted(glob.glob(os.path.join(SRC, '**', pat), recursive=True))
_fv = _find(f'p_{ARM}_s*_ep{EP}.npy')
if not _fv:
    print(f'khong thay p_{ARM}_s*_ep{EP}.npy trong {SRC}. Thu muc do dang co:')
    for _d in sorted(glob.glob(os.path.join(SRC, '*'))):
        _n = len(glob.glob(os.path.join(_d, '*.npy'))) if os.path.isdir(_d) else 0
        print(f'   {os.path.basename(_d)}{"/" if os.path.isdir(_d) else ""}'
              + (f'   {_n} file .npy' if _n else ''))
    _any = sorted(set(os.path.basename(f).split('_s')[0]
                      for f in glob.glob(os.path.join(SRC, '**', 'p_*.npy'), recursive=True)))
    print(f'\ncac arm thuc su co du doan: {_any}')
    _eps = sorted(set(int(f.rsplit("_ep", 1)[1][:-4])
                      for f in glob.glob(os.path.join(SRC, '**', f'p_{ARM}_s*.npy'), recursive=True)))
    print(f'cac epoch co cho `{ARM}`: {_eps}')
assert _fv, (f'khong thay du doan cho `{ARM}` ep{EP}. Xem danh sach vua in o tren roi sua ba dong '
             'SRC / ARM / EP o dau cell (SRC co the can them ten RUN_TAG).')
print(f'thu muc nguon: {os.path.dirname(_fv[0])}')
PV40 = np.mean([np.load(f) for f in _fv], 0)      # ensemble seed — 48 do +0.0103, khong ton GPU
assert PV40.shape == (len(I_VA53), len(LAB40)), f'hinh dang la {PV40.shape}'
print(f'nap {len(_fv)} seed (dev): {[os.path.basename(f) for f in _fv]}')

DEV_NEW = PV40[:, IDX26]
ap_new  = ap_all(YM[VA], DEV_NEW)
_old = float(D.mAP_anh.max()) if 'D' in dir() and hasattr(D, 'mAP_anh') else float('nan')
_has_old = _old == _old
print(f'\n=== DEV {int(VA.sum()):,} anh, {len(LAB26)} nhan ===')
print(f'  probe tot nhat (C3)   ' + (f'{_old:.4f}' if _has_old else '  —   (C3 chua chay)'))
print(f'  {ARM} (cat cot)  {np.nanmean(ap_new):.4f}'
      + (f'   {np.nanmean(ap_new)-_old:+.4f}' if _has_old else ''))

# ---------- TEST = 406 anh chuyen gia (cong 3 da chung minh nam trong split test) ----------
_ft = [f for e in range(EP-NLAST+1, EP+1) for f in _find(f'pte_{ARM}_s*_ep{e}.npy')]
if not _ft:
    print(f'\n!! khong thay pte_{ARM}_s*_ep*.npy -> nguon chua chay nhanh test. Bo qua phan 406.')
else:
    PT40 = np.mean([np.load(f) for f in _ft], 0)  # luat trung binh 3 epoch cuoi cua 48
    assert PT40.shape == (len(I_TE53), len(LAB40)), f'hinh dang la {PT40.shape}'
    _pos = {int(r): i for i, r in enumerate(I_TE53)}
    assert set(int(r) for r in T406) <= set(_pos), '406 anh khong nam tron trong mang pte'
    T_NEW = PT40[[_pos[int(r)] for r in T406]][:, IDX26]

    with open(os.path.join(T2_DIR, '_test_touches.jsonl'), 'a') as _f:
        _f.write(json.dumps(dict(n=len(T406), nguon='34/C7', arm=ARM,
                                 file=[os.path.basename(f) for f in _ft],
                                 muc_dich='bao cao')) + '\n')
    ap406 = ap_all(Y406, T_NEW)
    print(f'\n=== TEST 406 anh CHUYEN GIA, {len(LAB26)} nhan ===')
    print(f'  {ARM}  {np.nanmean(ap406):.4f}   | vo dich Task 2: 0.526')

    # --- san nhieu CUA CHINH TAP NAY. Bootstrap theo BENH NHAN: hai phim cung nguoi khong doc lap,
    #     lay mau theo ANH se cho khoang HEP GIA. ---
    _sub = ORDER.subject_id.to_numpy()[T406]; _uq = np.unique(_sub)
    _rs = np.random.default_rng(4); bs = []
    for _ in range(2000):
        pick = _rs.choice(_uq, size=len(_uq), replace=True)
        idx  = np.concatenate([np.where(_sub == u)[0] for u in pick])
        bs.append(np.nanmean(ap_all(Y406[idx], T_NEW[idx])))
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f'  CI95 [{lo:.4f}, {hi:.4f}]  rong {hi-lo:.4f}  -> san nhieu ~{(hi-lo)/2:.4f}')
    print(f'  = {(hi-lo)/2/0.0007:.0f}x san cua dev (+-0.0007). Hieu so duoi muc do KHONG doc duoc.')
    print('  ' + ('vo dich 0.526 NAM NGOAI khoang -> khoang cach la that'
                  if 0.526 > hi else
                  'vo dich 0.526 NAM TRONG khoang -> chua phan biet duoc voi ta'))

    _t = (pd.DataFrame(dict(label=LAB26, ap=ap406, n_pos=Y406.sum(0).astype(int)))
            .sort_values('ap', ascending=False))
    print(f'\n5 nhan tot nhat / kem nhat (n_pos = ca duong trong 406 anh):')
    print(_t.head(5).to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print(_t.tail(5).to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print(f'\n{int((_t.n_pos < 10).sum())}/{len(LAB26)} nhan co <10 ca duong -> AP cua chung gan nhu '
          'la nhieu thuan tuy, DUNG doc rieng le.')
    _t.to_csv(os.path.join(T2_DIR, f'per_label_{ARM}_406.csv'), index=False)